In [4]:
import pandas as pd
import numpy as np
import math

In [5]:
df_shot=pd.read_csv('shots.csv')
df_shot=df_shot[df_shot['shot_type']=='Open Play']
df_shot=df_shot[df_shot['shot_body_part']!='Head']

In [6]:
def calculateDistance(x,y):
    x_distance=120-x
    y_distance=0
    if (y<36):
        y_distance = 36-y
    elif (y>44):
        y_distance = y-44
    return np.sqrt(y_distance**2+x_distance**2)

def calculateAngle(x,y):
    g0 = [120, 44]
    p = [x, y]
    g1 = [120, 36]

    v0 = np.array(g0) - np.array(p)
    v1 = np.array(g1) - np.array(p)

    angle = math.atan2(np.linalg.det([v0,v1]),np.dot(v0,v1))
    return(abs(np.degrees(angle)))

def calculateDistanceShooterGk(x1,y1,x2,y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

In [7]:
df_shot['angle'] = df_shot.apply(lambda row:calculateAngle(row['x'], row['y']), axis=1)
df_shot['distance'] = df_shot.apply(lambda row:calculateDistance(row['x'], row['y']), axis=1)
df_shot['1on1'] = df_shot.apply(lambda row:1 if row['shot_one_on_one']==True else 0, axis=1)
df_shot['underPressure'] = df_shot.apply(lambda row:1 if row['under_pressure']==True else 0, axis=1)

df_shot['DistanceShooterGk'] = df_shot.apply(
    lambda row: calculateDistanceShooterGk(row['x'], row['y'], row['x_gk'], row['y_gk']) 
    if not row['y_gk']!=np.nan else np.nan, 
    axis=1
)
df_shot['DistanceGk'] = df_shot.apply(
    lambda row: calculateDistance(row['x_gk'], row['y_gk'])
    if not row['y_gk']!=np.nan else np.nan,
     axis=1
)
df_shot['minus'] = df_shot.apply(
    lambda row: row['x']-row['x_gk'], axis=1
)

In [8]:
df_shot.head()

,competition_id,season_id,match_id,location,shot_outcome,shot_statsbomb_xg,under_pressure,shot_type,shot_body_part,shot_one_on_one,...,x_gk,y_gk,goal,angle,distance,1on1,underPressure,DistanceShooterGk,DistanceGk,minus
0,9,281,3895302,"[89.2, 42.5]",Blocked,0.021272,True,Open Play,Left Foot,NaN,...,116.9,40.1,0,14.704957,30.800000,0,1,NaN,NaN,-27.7
1,9,281,3895302,"[97.5, 40.8]",Saved,0.048408,True,Open Play,Left Foot,NaN,...,117.8,40.4,0,20.137024,22.500000,0,1,NaN,NaN,-20.3
2,9,281,3895302,"[93.8, 34.8]",Off T,0.024202,True,Open Play,Right Foot,NaN,...,118.6,39.9,0,16.726070,26.227467,0,1,NaN,NaN,-24.8
3,9,281,3895302,"[93.0, 39.1]",Goal,0.031473,NaN,Open Play,Left Foot,NaN,...,118.6,39.9,1,16.835896,27.000000,0,0,NaN,NaN,-25.6
4,9,281,3895302,"[93.5, 40.1]",Goal,0.042012,NaN,Open Play,Right Foot,NaN,...,116.7,39.9,1,17.167008,26.500000,0,0,NaN,NaN,-23.2


In [7]:
df_shot.to_csv('shots_openplay_foot.csv', index=False)